# Retail Intelligence Platform - SQL Product and Category Analysis

This notebook builds the **product and category analytics layer** for the Retail Intelligence Platform using PostgreSQL.

## Objectives
- analyze product-level and category-level revenue
- identify top products and top categories
- calculate category contribution to sales
- evaluate average item price and order penetration by category
- export clean product outputs to `sql/outputs/`

## PostgreSQL tables used
- `olist_order_items`
- `olist_products`
- `product_category_name_translation`

## Reusable views created in this notebook
- `vw_product_sales`
- `vw_category_sales`

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

# --------------------------------------------------
# PostgreSQL connection
# --------------------------------------------------
DB_USER = "postgres"
DB_PASSWORD = quote_plus("1234")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "retail_intelligence"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("PostgreSQL engine created successfully.")

PostgreSQL engine created successfully.


## 1. Validate product table
We first confirm that product records exist and the database connection is working correctly.

In [2]:
product_test_query = """
SELECT COUNT(*) AS total_products
FROM olist_products;
"""

df_product_test = pd.read_sql(product_test_query, engine)
df_product_test

,total_products
0,32951


## 2. Create product and category analytical views

This section creates reusable views for:
- product-level sales performance
- category-level sales performance
- translated category names where available

In [14]:
create_product_views_sql = """
-- ------------------------------------------------------------
-- Product-level sales view
-- ------------------------------------------------------------
DROP VIEW IF EXISTS vw_product_sales;

CREATE VIEW vw_product_sales AS
SELECT
    oi.product_id,
    p.product_category_name,
    COALESCE(pct.product_category_name_english, p.product_category_name, 'Unknown') AS product_category_name_english,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(*) AS total_items_sold,
    SUM(oi.price) AS product_revenue,
    SUM(oi.freight_value) AS product_freight,
    AVG(oi.price) AS avg_item_price
FROM olist_order_items oi
LEFT JOIN olist_products p
    ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation pct
    ON p.product_category_name = pct.product_category_name
GROUP BY
    oi.product_id,
    p.product_category_name,
    COALESCE(pct.product_category_name_english, p.product_category_name, 'Unknown');


-- ------------------------------------------------------------
-- Category-level sales view
-- ------------------------------------------------------------
DROP VIEW IF EXISTS vw_category_sales;

CREATE VIEW vw_category_sales AS
SELECT
    COALESCE(pct.product_category_name_english, p.product_category_name, 'Unknown') AS category_name,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(*) AS total_items_sold,
    SUM(oi.price) AS category_revenue,
    SUM(oi.freight_value) AS category_freight,
    AVG(oi.price) AS avg_item_price
FROM olist_order_items oi
LEFT JOIN olist_products p
    ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation pct
    ON p.product_category_name = pct.product_category_name
GROUP BY
    COALESCE(pct.product_category_name_english, p.product_category_name, 'Unknown');
"""

with engine.begin() as conn:
    conn.exec_driver_sql(create_product_views_sql)

print("vw_product_sales and vw_category_sales created successfully.")

vw_product_sales and vw_category_sales created successfully.


## 3. Preview product sales view

In [15]:
product_sales_preview_query = """
SELECT *
FROM vw_product_sales
LIMIT 10;
"""

df_product_sales_preview = pd.read_sql(product_sales_preview_query, engine)
df_product_sales_preview

,product_id,product_category_name,product_category_name_english,total_orders,total_items_sold,product_revenue,product_freight,avg_item_price
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,perfumaria,1,1,101.65,18.59,101.65
1,00088930e925c41fd95ebfe695fd2655,automotivo,automotivo,1,1,129.90,13.93,129.90
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,cama_mesa_banho,1,1,229.00,13.10,229.00
3,000b8f95fcb9e0096488278317764d19,utilidades_domesticas,utilidades_domesticas,2,2,117.80,39.20,58.90
4,000d9be29b5207b54e86aa1b1ac54872,relogios_presentes,relogios_presentes,1,1,199.00,19.27,199.00
5,0011c512eb256aa0dbbb544d8dffcf6e,automotivo,automotivo,1,1,52.00,15.80,52.00
6,00126f27c813603687e6ce486d909d01,cool_stuff,cool_stuff,2,2,498.00,29.73,249.00
7,001795ec6f1b187d37335e1c4704762e,consoles_games,consoles_games,7,9,350.10,106.10,38.90
8,001b237c0e9bb435f2e54071129237e9,cama_mesa_banho,cama_mesa_banho,1,1,78.90,21.19,78.90
9,001b72dfd63e9833e8c02742adf472e3,moveis_decoracao,moveis_decoracao,12,14,489.86,174.68,34.99


## 4. Product and category KPI summary

This section calculates high-level product and category KPIs:
- total products sold
- total categories
- total product revenue
- average revenue per product
- average items sold per product

In [16]:
product_kpi_query = """
SELECT
    COUNT(DISTINCT product_id) AS total_products_sold,
    COUNT(DISTINCT product_category_name_english) AS total_categories,
    ROUND(SUM(product_revenue)::numeric, 2) AS total_product_revenue,
    ROUND(AVG(product_revenue)::numeric, 2) AS avg_revenue_per_product,
    ROUND(AVG(total_items_sold)::numeric, 2) AS avg_items_sold_per_product,
    ROUND(AVG(avg_item_price)::numeric, 2) AS avg_product_price
FROM vw_product_sales;
"""

df_product_kpis = pd.read_sql(product_kpi_query, engine)
df_product_kpis

,total_products_sold,total_categories,total_product_revenue,avg_revenue_per_product,avg_items_sold_per_product,avg_product_price
0,32951,74,13591643.7,412.48,3.42,145.3


## 5. Top products by revenue
This section identifies the highest revenue-generating products.

In [17]:
top_products_revenue_query = """
SELECT
    product_id,
    product_category_name_english,
    total_orders,
    total_items_sold,
    ROUND(product_revenue::numeric, 2) AS product_revenue,
    ROUND(avg_item_price::numeric, 2) AS avg_item_price
FROM vw_product_sales
ORDER BY product_revenue DESC
LIMIT 25;
"""

df_top_products_revenue = pd.read_sql(top_products_revenue_query, engine)
df_top_products_revenue

,product_id,product_category_name_english,total_orders,total_items_sold,product_revenue,avg_item_price
0,bb50f2e236e5eea0100680137654686c,beleza_saude,187,195,63885.00,327.62
1,6cdd53843498f92890544667809f1595,beleza_saude,151,156,54730.20,350.83
2,d6160fb7873f184099d9bc95e30376af,pcs,35,35,48899.34,1397.12
3,d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,323,343,47214.51,137.65
4,99a4788cb24856965c36a24e339b6058,cama_mesa_banho,467,488,43025.56,88.17
5,3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,255,274,41082.60,149.94
6,25c38557cf793876c5abdd5931f922db,bebes,38,38,38907.32,1023.88
7,5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,63,37733.90,598.95
8,53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,306,323,37683.42,116.67
9,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,431,527,37608.90,71.36


## 6. Top products by order volume
This section identifies products that appeared in the highest number of distinct orders.

In [18]:
top_products_orders_query = """
SELECT
    product_id,
    product_category_name_english,
    total_orders,
    total_items_sold,
    ROUND(product_revenue::numeric, 2) AS product_revenue
FROM vw_product_sales
ORDER BY total_orders DESC, product_revenue DESC
LIMIT 25;
"""

df_top_products_orders = pd.read_sql(top_products_orders_query, engine)
df_top_products_orders

,product_id,product_category_name_english,total_orders,total_items_sold,product_revenue
0,99a4788cb24856965c36a24e339b6058,cama_mesa_banho,467,488,43025.56
1,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,431,527,37608.90
2,422879e10f46682990de24d770e7f83d,ferramentas_jardim,352,484,26577.22
3,d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,323,343,47214.51
4,389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,311,392,21440.59
5,53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,306,323,37683.42
6,368c6c730842d78016ad823897a372db,ferramentas_jardim,291,388,21056.80
7,53759a2ecddad2bb87a079a1f1519f73,ferramentas_jardim,287,373,20387.20
8,154e7e31ebfa092203795c972e5804a6,beleza_saude,269,281,6325.19
9,2b4609f8948be18874494203496bc318,beleza_saude,259,260,22717.22


## 7. Category revenue analysis
This section summarizes revenue, orders, and item volume by category.

In [19]:
category_revenue_query = """
SELECT
    category_name,
    total_orders,
    total_items_sold,
    ROUND(category_revenue::numeric, 2) AS category_revenue,
    ROUND(category_freight::numeric, 2) AS category_freight,
    ROUND(avg_item_price::numeric, 2) AS avg_item_price
FROM vw_category_sales
ORDER BY category_revenue DESC;
"""

df_category_revenue = pd.read_sql(category_revenue_query, engine)
df_category_revenue

,category_name,total_orders,total_items_sold,category_revenue,category_freight,avg_item_price
0,beleza_saude,8836,9670,1258681.34,182566.73,130.16
1,relogios_presentes,5624,5991,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,11115,1036988.68,204693.04,93.30
3,esporte_lazer,7720,8641,988048.97,168607.51,114.34
4,informatica_acessorios,6689,7827,911954.32,147318.08,116.51
...,...,...,...,...,...,...
69,flores,29,33,1110.04,488.87,33.64
70,casa_conforto_2,24,30,760.27,410.31,25.34
71,cds_dvds_musicais,12,14,730.00,224.99,52.14
72,fashion_roupa_infanto_juvenil,8,8,569.85,95.51,71.23


## 8. Category contribution to total revenue
This section calculates how much each category contributes to overall product revenue.

In [20]:
category_share_query = """
WITH total_rev AS (
    SELECT SUM(category_revenue) AS grand_total_revenue
    FROM vw_category_sales
)
SELECT
    c.category_name,
    c.total_orders,
    c.total_items_sold,
    ROUND(c.category_revenue::numeric, 2) AS category_revenue,
    ROUND(
        100.0 * c.category_revenue / NULLIF(t.grand_total_revenue, 0),
        2
    ) AS revenue_share_pct
FROM vw_category_sales c
CROSS JOIN total_rev t
ORDER BY category_revenue DESC;
"""

df_category_share = pd.read_sql(category_share_query, engine)
df_category_share

,category_name,total_orders,total_items_sold,category_revenue,revenue_share_pct
0,beleza_saude,8836,9670,1258681.34,9.26
1,relogios_presentes,5624,5991,1205005.68,8.87
2,cama_mesa_banho,9417,11115,1036988.68,7.63
3,esporte_lazer,7720,8641,988048.97,7.27
4,informatica_acessorios,6689,7827,911954.32,6.71
...,...,...,...,...,...
69,flores,29,33,1110.04,0.01
70,casa_conforto_2,24,30,760.27,0.01
71,cds_dvds_musicais,12,14,730.00,0.01
72,fashion_roupa_infanto_juvenil,8,8,569.85,0.00


## 9. Category order penetration
This section shows how many orders each category participated in and the average items sold.

In [21]:
category_penetration_query = """
SELECT
    category_name,
    total_orders,
    total_items_sold,
    ROUND(
        total_items_sold::numeric / NULLIF(total_orders, 0),
        2
    ) AS avg_items_per_order,
    ROUND(avg_item_price::numeric, 2) AS avg_item_price
FROM vw_category_sales
ORDER BY total_orders DESC;
"""

df_category_penetration = pd.read_sql(category_penetration_query, engine)
df_category_penetration

,category_name,total_orders,total_items_sold,avg_items_per_order,avg_item_price
0,cama_mesa_banho,9417,11115,1.18,93.30
1,beleza_saude,8836,9670,1.09,130.16
2,esporte_lazer,7720,8641,1.12,114.34
3,informatica_acessorios,6689,7827,1.17,116.51
4,moveis_decoracao,6449,8334,1.29,87.56
...,...,...,...,...,...
69,la_cuisine,13,14,1.08,146.79
70,cds_dvds_musicais,12,14,1.17,52.14
71,pc_gamer,8,9,1.13,171.77
72,fashion_roupa_infanto_juvenil,8,8,1.00,71.23


## 10. Product segmentation
This section groups products into simple revenue-based segments.

In [22]:
product_segment_query = """
SELECT
    CASE
        WHEN product_revenue < 100 THEN 'Low'
        WHEN product_revenue < 500 THEN 'Medium'
        WHEN product_revenue < 2000 THEN 'High'
        ELSE 'Top'
    END AS product_segment,
    COUNT(*) AS product_count,
    ROUND(AVG(product_revenue)::numeric, 2) AS avg_product_revenue,
    ROUND(AVG(total_orders)::numeric, 2) AS avg_orders
FROM vw_product_sales
GROUP BY
    CASE
        WHEN product_revenue < 100 THEN 'Low'
        WHEN product_revenue < 500 THEN 'Medium'
        WHEN product_revenue < 2000 THEN 'High'
        ELSE 'Top'
    END
ORDER BY product_count DESC;
"""

df_product_segments = pd.read_sql(product_segment_query, engine)
df_product_segments

,product_segment,product_count,avg_product_revenue,avg_orders
0,Medium,13998,229.14,2.27
1,Low,13463,53.06,1.24
2,High,4426,944.18,5.91
3,Top,1064,5160.60,26.08


## 11. Export product outputs
Save all product analysis outputs into the project `sql/outputs/` folder.

In [23]:
OUTPUT_DIR = Path(r"C:\Users\divya\Retail-Intelligence-Platform\sql\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_product_kpis.to_csv(OUTPUT_DIR / "product_summary.csv", index=False)
df_top_products_revenue.to_csv(OUTPUT_DIR / "top_products_revenue.csv", index=False)
df_top_products_orders.to_csv(OUTPUT_DIR / "top_products_orders.csv", index=False)
df_category_revenue.to_csv(OUTPUT_DIR / "category_summary.csv", index=False)
df_category_share.to_csv(OUTPUT_DIR / "category_revenue_share.csv", index=False)
df_category_penetration.to_csv(OUTPUT_DIR / "category_penetration_summary.csv", index=False)
df_product_segments.to_csv(OUTPUT_DIR / "product_segment_summary.csv", index=False)

print("Product SQL outputs exported successfully.")
print(f"Saved to: {OUTPUT_DIR}")

Product SQL outputs exported successfully.
Saved to: C:\Users\divya\Retail-Intelligence-Platform\sql\outputs


## 12. Summary

This notebook created the SQL product and category analytics layer for the Retail Intelligence Platform.

### Outputs generated
- `product_summary.csv`
- `top_products_revenue.csv`
- `top_products_orders.csv`
- `category_summary.csv`
- `category_revenue_share.csv`
- `category_penetration_summary.csv`
- `product_segment_summary.csv`

### Next notebook
`05_sql_reviews_delivery.ipynb`